# QOS Real-Dataset Experiments
**Tommaso R. Marena (2026)**

Runs all five real-dataset experiments from the paper (Section 6):
- **IMDb** — Binary sentiment classification (~60 logical qubits)
- **20 Newsgroups** — Multi-class topic classification (~60 logical qubits)
- **PBMC68k** — Single-cell RNA binary classification (~50 logical qubits)
- **Dorothea** — Drug discovery binary classification (~60 logical qubits)
- **Splice** — DNA junction binary classification (~40 logical qubits)

Each dataset produces:
- `*_size_vs_accuracy.pdf` — Machine size vs classification accuracy (Quantum / Sparse / Streaming)
- `*_size_vs_variance.pdf` — Machine size vs PCA variance recovery
- `*_size_vs_accuracy.json` — Raw data for paper tables

> **Runtime:** ~2-4 hrs on A100, ~8-12 hrs on T4. Use Colab Pro+ with A100 for paper-quality runs.
> All outputs are auto-saved to Google Drive if mounted.

## 0. Setup

In [ ]:
# Install the QOS package and all dependencies
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'quantum-oracle-sketching[dev,noise,kernel] @ git+https://github.com/Tommaso-R-Marena/quantum_oracle_sketching.git'],
    capture_output=True, text=True
)
print(result.stdout[-1000:] if result.stdout else '')
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
else:
    print('Install successful.')

In [ ]:
# Mount Google Drive to persist results (optional but recommended)
import os
MOUNT_DRIVE = True  # Set False to save only to /content/results/

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/qos_real_datasets'
else:
    OUTPUT_DIR = '/content/results'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
# Global configuration
FAST_MODE  = False
SAVE_FIGS  = True
SHOW_FIGS  = True
CV_FOLDS   = 5

# Per-dataset regularization — must match Zhao et al. (2025) Appendix A exactly
CLF_ALPHA_PER_DATASET = {
    'imdb':     10.0,    # Zhao Fig. 2a
    'news20':    1.0,    # Zhao Fig. 4a (Appendix A)
    'pbmc68k': 200.0,    # Zhao Fig. 2b
    'dorothea': 200.0,   # Zhao Fig. 4b (Appendix A)
    'splice':   10.0,    # Novel Marena 2026 — no Zhao reference
}
FAST_SWEEP_LEN = 10

import jax
print('JAX devices:', jax.devices())
print(f'Mode: {"FAST" if FAST_MODE else "PAPER QUALITY"}')
print('CLF_ALPHA per dataset:', CLF_ALPHA_PER_DATASET)


In [ ]:
import sys, os, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'figure.dpi': 150, 'font.size': 10})

# Inline helper: display a saved PDF as PNG for Colab preview
from IPython.display import display, Image
from pdf2image import convert_from_path

def show_pdf(path):
    try:
        imgs = convert_from_path(path, dpi=150)
        for img in imgs:
            display(img)
    except Exception:
        print(f'PDF saved to {path} (install pdf2image + poppler to preview inline)')

# Install pdf2image silently
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pdf2image'], capture_output=True)
subprocess.run(['apt-get', 'install', '-qq', 'poppler-utils'], capture_output=True)
print('Shared imports ready.')

In [ ]:
# ============================================================
# ZHAO ET AL. (2025) REFERENCE VALUES — Figures 2 and 4
# Read from paper at the Pareto-optimal operating point of QOS
# (highest accuracy achieved by QOS; classical sizes at same accuracy)
# ============================================================
ZHAO_REFERENCE = {
    'imdb': {
        'quantum_qubits':           57,
        'peak_accuracy':            0.868,
        'classical_streaming_size': 1.2e5,
        'classical_sparse_size':    3.1e6,
        'advantage_oom_range':      (4, 6),
        'regularization_alpha':     10.0,
        'cv_folds':                 5,
        'fig_reference':            'Figure 2a',
    },
    'pbmc68k': {
        'quantum_qubits':           50,
        'peak_accuracy':            0.920,
        'classical_streaming_size': 2.5e5,
        'classical_sparse_size':    1.8e6,
        'advantage_oom_range':      (4, 6),
        'regularization_alpha':     200.0,
        'cv_folds':                 5,
        'fig_reference':            'Figure 2b',
    },
    'news20': {
        'quantum_qubits':           58,
        'peak_accuracy':            0.790,
        'classical_streaming_size': 8.0e4,
        'classical_sparse_size':    9.0e5,
        'advantage_oom_range':      (4, 5),
        'regularization_alpha':     1.0,
        'cv_folds':                 5,
        'fig_reference':            'Figure 4a (Appendix A)',
    },
    'dorothea': {
        'quantum_qubits':           57,
        'peak_accuracy':            0.910,
        'classical_streaming_size': 1.0e5,
        'classical_sparse_size':    1.4e5,
        'advantage_oom_range':      (3, 5),
        'regularization_alpha':     200.0,
        'cv_folds':                 5,
        'fig_reference':            'Figure 4b (Appendix A)',
    },
    'splice': {
        'quantum_qubits':  None,
        'note': 'Splice not studied in Zhao et al. (2025). Results are novel (Marena 2026).',
    },
}

# ============================================================
# A1–A3 EQUATION VERIFICATION — crashes loudly if implementation drifts
# ============================================================
def _verify_zhao_equations():
    import scipy.sparse as sp_
    import numpy as np
    import math
    from qos.experiments.real_datasets.shared import compute_space_metrics

    rng = np.random.default_rng(42)
    rows, cols = [], []
    for i in range(100):
        c = rng.choice(50, size=5, replace=False)
        rows.extend([i]*5); cols.extend(c.tolist())
    X = sp_.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(100, 50))
    N, D, N_nnz = 100, 50, X.nnz
    s = int(max(np.diff(X.tocsr().indptr).max(), np.diff(X.tocsc().indptr).max()))

    m = compute_space_metrics(X)

    # A1
    assert m['space_sparse'] == N_nnz,    f"A1 FAIL: space_sparse={m['space_sparse']} != {N_nnz}"
    assert m['space_qram']   == N_nnz,    f"A1 FAIL: space_qram={m['space_qram']} != {N_nnz}"
    # A2
    assert m['space_streaming'] == D,     f"A2 FAIL: space_streaming={m['space_streaming']} != {D}"
    # A3
    expected = 2*math.ceil(math.log2(N+2*D)) + math.ceil(math.log2(s+1)) + 3 + 1
    assert m['space_quantum'] == expected, f"A3 FAIL: space_quantum={m['space_quantum']} != {expected}"

    print("✅ Zhao et al. (2025) Equations A1, A2, A3 verified.")
    print(f"   N={N}, D={D}, N_nnz={N_nnz}, s={s}")
    print(f"   A1: S_sparse = S_QRAM = {N_nnz}")
    print(f"   A2: S_streaming = {D}")
    print(f"   A3: S_QOS = 2⌈log₂({N}+2×{D})⌉ + ⌈log₂({s}+1)⌉ + 3 + 1 = {expected} qubits")

_verify_zhao_equations()

def compare_to_zhao(dataset_name, results, zhao_ref):
    """
    Rigorous comparison of QOS results against Zhao et al. (2025).
    Reproduction criteria: |Δacc| < 2%, |Δqubits|/zhao_qubits < 20%.
    """
    if zhao_ref.get('quantum_qubits') is None:
        return {'match_status': 'NOT_AVAILABLE', 'note': zhao_ref.get('note', '')}

    our_peak_acc = max(results['accuracies_mean'])
    our_peak_idx = results['accuracies_mean'].index(our_peak_acc)
    our_qubits   = results['space_quantum'][our_peak_idx]
    our_stream   = results['space_streaming'][our_peak_idx]

    acc_delta   = our_peak_acc - zhao_ref['peak_accuracy']
    qubit_delta = our_qubits   - zhao_ref['quantum_qubits']
    our_oom     = np.log10(max(our_stream, 1) / max(our_qubits, 1))

    acc_match   = abs(acc_delta)   < 0.02
    qubit_match = abs(qubit_delta) / zhao_ref['quantum_qubits'] < 0.20

    if acc_match and qubit_match:
        status = 'REPRODUCED'
    elif our_peak_acc > zhao_ref['peak_accuracy'] + 0.02 or our_qubits < zhao_ref['quantum_qubits'] * 0.80:
        status = 'IMPROVED'
    else:
        status = 'DEGRADED'

    return {
        'match_status':          status,
        'reproduced':            status in ('REPRODUCED', 'IMPROVED'),
        'our_peak_accuracy':     round(our_peak_acc, 4),
        'zhao_peak_accuracy':    zhao_ref['peak_accuracy'],
        'accuracy_delta':        round(acc_delta, 4),
        'our_qubits_at_peak':    int(our_qubits),
        'zhao_qubits':           zhao_ref['quantum_qubits'],
        'qubit_delta':           int(qubit_delta),
        'our_advantage_oom':     round(our_oom, 2),
        'zhao_advantage_oom_range': zhao_ref['advantage_oom_range'],
        'fig_reference':         zhao_ref['fig_reference'],
    }

print("compare_to_zhao() loaded.")


## 1. IMDb — Binary Sentiment Classification

In [ ]:
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from qos.experiments.real_datasets.shared import (
    run_classification_experiment, run_pca_experiment,
    plot_experiment_results, plot_pca_results,
)
from qos.experiments.real_datasets.imdb.imdb_utils import load_imdb_data

IMDB_MIN_DFS = [
    2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 14, 16, 19, 21, 24, 28, 32, 36, 42, 48,
    55, 62, 71, 81, 93, 106, 122, 139, 159, 181, 207, 236, 270, 308, 352, 402,
    459, 524, 599, 684, 781, 891, 1018, 1162, 1327, 1515, 1730, 1976, 2256,
    2576, 2941, 3358, 3835, 4379, 5000,
]
if FAST_MODE:
    IMDB_MIN_DFS = IMDB_MIN_DFS[::max(1, len(IMDB_MIN_DFS)//FAST_SWEEP_LEN)]

print('Loading IMDb...')
X_raw, y = load_imdb_data()
print(f'Loaded {len(X_raw)} reviews, {len(set(y))} classes')

def imdb_filter(X_raw, min_df):
    vec = TfidfVectorizer(min_df=min_df, stop_words='english')
    X = vec.fit_transform(X_raw)
    X.eliminate_zeros()
    return X, {'vocab_size': X.shape[1]}

prefix = os.path.join(OUTPUT_DIR, 'imdb')
t0 = time.time()

print('Running IMDb classification sweep...')
imdb_clf = run_classification_experiment(
    X_raw, y, sweep_values=IMDB_MIN_DFS, filter_fn=imdb_filter,
    dataset_name='IMDb', output_prefix=prefix,
    clf_alpha=CLF_ALPHA_PER_DATASET['imdb'], cv_folds=CV_FOLDS,
)
plot_experiment_results(
    imdb_clf, title='IMDb: Binary classification',
    output_prefix=prefix, xlabel='Accuracy',
    xticks=[0.70, 0.75, 0.80, 0.85, 0.90],
    xtick_labels=['70%','75%','80%','85%','90%'],
    xlim=(0.69, 0.91), ylim=(1e1, 1e7),
    text_positions={'sparse':(0.70,4e6),'streaming':(0.88,9e4),'quantum':(0.90,1.9e1)},
)
if SHOW_FIGS: show_pdf(prefix + '_size_vs_accuracy.pdf')

# --- Zhao et al. (2025) comparison ---
_comp = compare_to_zhao('imdb', imdb_clf, ZHAO_REFERENCE['imdb'])
print(f"\n{'='*60}")
print(f"ZHAO ET AL. COMPARISON — IMDB ({ZHAO_REFERENCE['imdb']['fig_reference']})")
print(f"{'='*60}")
print(f"Status:         {_comp['match_status']}")
print(f"Accuracy:       ours={_comp['our_peak_accuracy']:.3f}  Zhao={_comp['zhao_peak_accuracy']:.3f}  Δ={_comp['accuracy_delta']:+.3f}")
print(f"Qubits:         ours={_comp['our_qubits_at_peak']}  Zhao={_comp['zhao_qubits']}  Δ={_comp['qubit_delta']:+d}")
print(f"Advantage:      ours={_comp['our_advantage_oom']:.1f} OOM  Zhao={_comp['zhao_advantage_oom_range'][0]}–{_comp['zhao_advantage_oom_range'][1]} OOM")
print(f"Reproduced:     {'✅ YES' if _comp['reproduced'] else '❌ NO — investigate before submission'}")
_cpath = os.path.join(OUTPUT_DIR, 'imdb_zhao_comparison.json')
with open(_cpath, 'w') as _f: json.dump(_comp, _f, indent=2)
print(f"Saved: {_cpath}")

print('Running IMDb PCA sweep...')
imdb_pca = run_pca_experiment(
    X_raw, y, sweep_values=IMDB_MIN_DFS, filter_fn=imdb_filter,
    dataset_name='IMDb', output_prefix=prefix, n_components=2,
)
plot_pca_results(imdb_pca, title='IMDb: Dimension reduction',
    output_prefix=prefix, ylim=(1e1, 1e7))
if SHOW_FIGS: show_pdf(prefix + '_size_vs_variance.pdf')
print(f'IMDb done in {time.time()-t0:.0f}s')


## 2. 20 Newsgroups — Multi-class Topic Classification

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from qos.experiments.real_datasets.shared import (
    run_classification_experiment, run_pca_experiment,
    plot_experiment_results, plot_pca_results,
)

NEWS20_MIN_DFS = [
    2, 3, 4, 5, 6, 8, 10, 12, 15, 18, 21, 26, 31, 37, 44, 53, 63,
    76, 91, 109, 130, 156, 187, 224, 268, 321, 385, 461, 553, 662,
    793, 950, 1138, 1363, 1633, 1956, 2342, 2806, 3361, 4027, 4826, 5000,
]
if FAST_MODE:
    NEWS20_MIN_DFS = NEWS20_MIN_DFS[::max(1, len(NEWS20_MIN_DFS)//FAST_SWEEP_LEN)]

print('Loading 20 Newsgroups...')
data = fetch_20newsgroups(subset='all', remove=('headers','footers','quotes'))
X_raw_news, y_news = data.data, data.target
print(f'Loaded {len(X_raw_news)} documents, {len(set(y_news))} classes')

def news20_filter(X_raw, min_df):
    vec = TfidfVectorizer(min_df=min_df, stop_words='english')
    X = vec.fit_transform(X_raw)
    X.eliminate_zeros()
    return X, {'vocab_size': X.shape[1]}

prefix = os.path.join(OUTPUT_DIR, 'news20')
t0 = time.time()

print('Running 20 Newsgroups classification sweep...')
news_clf = run_classification_experiment(
    X_raw_news, y_news, sweep_values=NEWS20_MIN_DFS, filter_fn=news20_filter,
    dataset_name='20 Newsgroups', output_prefix=prefix,
    clf_alpha=CLF_ALPHA_PER_DATASET['news20'], cv_folds=CV_FOLDS,
)
plot_experiment_results(
    news_clf, title='20 Newsgroups: Multi-class',
    output_prefix=prefix, xlabel='Accuracy',
    xticks=[0.55, 0.65, 0.75, 0.85],
    xtick_labels=['55%','65%','75%','85%'],
    xlim=(0.50, 0.88), ylim=(1e1, 1e7),
)
if SHOW_FIGS: show_pdf(prefix + '_size_vs_accuracy.pdf')

# --- Zhao et al. (2025) comparison ---
_comp = compare_to_zhao('news20', news_clf, ZHAO_REFERENCE['news20'])
print(f"\n{'='*60}")
print(f"ZHAO ET AL. COMPARISON — 20 NEWSGROUPS ({ZHAO_REFERENCE['news20']['fig_reference']})")
print(f"{'='*60}")
print(f"Status:         {_comp['match_status']}")
print(f"Accuracy:       ours={_comp['our_peak_accuracy']:.3f}  Zhao={_comp['zhao_peak_accuracy']:.3f}  Δ={_comp['accuracy_delta']:+.3f}")
print(f"Qubits:         ours={_comp['our_qubits_at_peak']}  Zhao={_comp['zhao_qubits']}  Δ={_comp['qubit_delta']:+d}")
print(f"Advantage:      ours={_comp['our_advantage_oom']:.1f} OOM  Zhao={_comp['zhao_advantage_oom_range'][0]}–{_comp['zhao_advantage_oom_range'][1]} OOM")
print(f"Reproduced:     {'✅ YES' if _comp['reproduced'] else '❌ NO — investigate before submission'}")
_cpath = os.path.join(OUTPUT_DIR, 'news20_zhao_comparison.json')
with open(_cpath, 'w') as _f: json.dump(_comp, _f, indent=2)
print(f"Saved: {_cpath}")

print('Running 20 Newsgroups PCA sweep...')
news_pca = run_pca_experiment(
    X_raw_news, y_news, sweep_values=NEWS20_MIN_DFS, filter_fn=news20_filter,
    dataset_name='20 Newsgroups', output_prefix=prefix, n_components=2,
)
plot_pca_results(news_pca, title='20 Newsgroups: Dimension reduction',
    output_prefix=prefix, ylim=(1e1, 1e7))
if SHOW_FIGS: show_pdf(prefix + '_size_vs_variance.pdf')
print(f'20 Newsgroups done in {time.time()-t0:.0f}s')


## 3. PBMC68k — Single-Cell RNA Binary Classification

In [ ]:
# PBMC68k requires scanpy + anndata
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scanpy', 'anndata'], capture_output=True)
print('scanpy installed.')

In [ ]:
import scanpy as sc
from qos.experiments.real_datasets.pbmc68k.pbmc_utils import load_pbmc68k
from qos.experiments.real_datasets.shared import (
    run_classification_experiment, run_pca_experiment,
    plot_experiment_results, plot_pca_results,
)

PBMC_MIN_COUNTS = [
    1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 18, 21, 25, 30, 36, 43,
    51, 61, 73, 87, 104, 124, 148, 177, 211, 252, 301, 360, 430, 514,
    614, 734, 877, 1048, 1252, 1496, 1788, 2137, 2554, 3053, 3650, 5000,
]
if FAST_MODE:
    PBMC_MIN_COUNTS = PBMC_MIN_COUNTS[::max(1, len(PBMC_MIN_COUNTS)//FAST_SWEEP_LEN)]

print('Loading PBMC68k (auto-downloads ~600MB on first run)...')
adata, y_pbmc = load_pbmc68k()
X_pbmc = adata.X  # scipy sparse
print(f'Loaded {X_pbmc.shape[0]} cells x {X_pbmc.shape[1]} genes, {len(set(y_pbmc))} classes')

def pbmc_filter(X, min_count):
    gene_counts = np.asarray(X.sum(axis=0)).ravel()
    mask = gene_counts >= min_count
    X_f = X[:, mask]
    if X_f.shape[1] == 0:
        return None, {}
    return X_f, {'num_genes': int(mask.sum())}

prefix = os.path.join(OUTPUT_DIR, 'pbmc68k')
t0 = time.time()

print('Running PBMC68k classification sweep...')
pbmc_clf = run_classification_experiment(
    X_pbmc, y_pbmc, sweep_values=PBMC_MIN_COUNTS, filter_fn=pbmc_filter,
    dataset_name='PBMC68k', output_prefix=prefix,
    clf_alpha=CLF_ALPHA_PER_DATASET['pbmc68k'], cv_folds=CV_FOLDS,
)
plot_experiment_results(
    pbmc_clf, title='PBMC68k: Cell-type classification',
    output_prefix=prefix, xlabel='Accuracy',
    xticks=[0.80, 0.85, 0.90, 0.95],
    xtick_labels=['80%','85%','90%','95%'],
    xlim=(0.78, 0.97), ylim=(1e1, 1e6),
)
if SHOW_FIGS: show_pdf(prefix + '_size_vs_accuracy.pdf')

# --- Zhao et al. (2025) comparison ---
_comp = compare_to_zhao('pbmc68k', pbmc_clf, ZHAO_REFERENCE['pbmc68k'])
print(f"\n{'='*60}")
print(f"ZHAO ET AL. COMPARISON — PBMC68K ({ZHAO_REFERENCE['pbmc68k']['fig_reference']})")
print(f"{'='*60}")
print(f"Status:         {_comp['match_status']}")
print(f"Accuracy:       ours={_comp['our_peak_accuracy']:.3f}  Zhao={_comp['zhao_peak_accuracy']:.3f}  Δ={_comp['accuracy_delta']:+.3f}")
print(f"Qubits:         ours={_comp['our_qubits_at_peak']}  Zhao={_comp['zhao_qubits']}  Δ={_comp['qubit_delta']:+d}")
print(f"Advantage:      ours={_comp['our_advantage_oom']:.1f} OOM  Zhao={_comp['zhao_advantage_oom_range'][0]}–{_comp['zhao_advantage_oom_range'][1]} OOM")
print(f"Reproduced:     {'✅ YES' if _comp['reproduced'] else '❌ NO — investigate before submission'}")
_cpath = os.path.join(OUTPUT_DIR, 'pbmc68k_zhao_comparison.json')
with open(_cpath, 'w') as _f: json.dump(_comp, _f, indent=2)
print(f"Saved: {_cpath}")

print('Running PBMC68k PCA sweep...')
pbmc_pca = run_pca_experiment(
    X_pbmc, y_pbmc, sweep_values=PBMC_MIN_COUNTS, filter_fn=pbmc_filter,
    dataset_name='PBMC68k', output_prefix=prefix, n_components=2,
)
plot_pca_results(pbmc_pca, title='PBMC68k: Dimension reduction',
    output_prefix=prefix, ylim=(1e1, 1e6))
if SHOW_FIGS: show_pdf(prefix + '_size_vs_variance.pdf')
print(f'PBMC68k done in {time.time()-t0:.0f}s')


## 4. Dorothea — Drug Discovery Binary Classification

In [ ]:
from qos.experiments.real_datasets.dorothea.dorothea_utils import load_dorothea
from qos.experiments.real_datasets.shared import (
    run_classification_experiment, run_pca_experiment,
    plot_experiment_results, plot_pca_results,
)

DOROTHEA_THRESHOLDS = [
    1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 14, 17, 20, 24, 28, 33, 39,
    46, 55, 65, 77, 91, 108, 128, 152, 180, 213, 252, 299, 354,
    420, 497, 589, 698, 827, 980, 1161, 1375, 1629, 1930, 2286, 2709,
]
if FAST_MODE:
    DOROTHEA_THRESHOLDS = DOROTHEA_THRESHOLDS[::max(1, len(DOROTHEA_THRESHOLDS)//FAST_SWEEP_LEN)]

print('Loading Dorothea (auto-downloads from UCI if not cached)...')
X_dor, y_dor = load_dorothea()
print(f'Loaded {X_dor.shape[0]} samples x {X_dor.shape[1]} features')

def dorothea_filter(X, threshold):
    col_counts = np.asarray(X.sum(axis=0)).ravel()
    mask = col_counts >= threshold
    X_f = X[:, mask]
    if X_f.shape[1] == 0:
        return None, {}
    return X_f, {'num_features': int(mask.sum())}

prefix = os.path.join(OUTPUT_DIR, 'dorothea')
t0 = time.time()

print('Running Dorothea classification sweep...')
dor_clf = run_classification_experiment(
    X_dor, y_dor, sweep_values=DOROTHEA_THRESHOLDS, filter_fn=dorothea_filter,
    dataset_name='Dorothea', output_prefix=prefix,
    clf_alpha=CLF_ALPHA_PER_DATASET['dorothea'], cv_folds=CV_FOLDS,
)
plot_experiment_results(
    dor_clf, title='Dorothea: Drug discovery',
    output_prefix=prefix, xlabel='Accuracy',
    xticks=[0.80, 0.85, 0.90, 0.95],
    xtick_labels=['80%','85%','90%','95%'],
    xlim=(0.78, 0.97), ylim=(1e1, 1e6),
)
if SHOW_FIGS: show_pdf(prefix + '_size_vs_accuracy.pdf')

# --- Zhao et al. (2025) comparison ---
_comp = compare_to_zhao('dorothea', dor_clf, ZHAO_REFERENCE['dorothea'])
print(f"\n{'='*60}")
print(f"ZHAO ET AL. COMPARISON — DOROTHEA ({ZHAO_REFERENCE['dorothea']['fig_reference']})")
print(f"{'='*60}")
print(f"Status:         {_comp['match_status']}")
print(f"Accuracy:       ours={_comp['our_peak_accuracy']:.3f}  Zhao={_comp['zhao_peak_accuracy']:.3f}  Δ={_comp['accuracy_delta']:+.3f}")
print(f"Qubits:         ours={_comp['our_qubits_at_peak']}  Zhao={_comp['zhao_qubits']}  Δ={_comp['qubit_delta']:+d}")
print(f"Advantage:      ours={_comp['our_advantage_oom']:.1f} OOM  Zhao={_comp['zhao_advantage_oom_range'][0]}–{_comp['zhao_advantage_oom_range'][1]} OOM")
print(f"Reproduced:     {'✅ YES' if _comp['reproduced'] else '❌ NO — investigate before submission'}")
_cpath = os.path.join(OUTPUT_DIR, 'dorothea_zhao_comparison.json')
with open(_cpath, 'w') as _f: json.dump(_comp, _f, indent=2)
print(f"Saved: {_cpath}")

print('Running Dorothea PCA sweep...')
dor_pca = run_pca_experiment(
    X_dor, y_dor, sweep_values=DOROTHEA_THRESHOLDS, filter_fn=dorothea_filter,
    dataset_name='Dorothea', output_prefix=prefix, n_components=2,
)
plot_pca_results(dor_pca, title='Dorothea: Dimension reduction',
    output_prefix=prefix, ylim=(1e1, 1e6))
if SHOW_FIGS: show_pdf(prefix + '_size_vs_variance.pdf')
print(f'Dorothea done in {time.time()-t0:.0f}s')


## 5. Splice — DNA Junction Binary Classification

In [ ]:
from qos.experiments.real_datasets.splice.splice_utils import load_splice
from qos.experiments.real_datasets.shared import (
    run_classification_experiment, run_pca_experiment,
    plot_experiment_results, plot_pca_results,
)

SPLICE_THRESHOLDS = list(range(1, 61))  # feature-occurrence threshold
if FAST_MODE:
    SPLICE_THRESHOLDS = SPLICE_THRESHOLDS[::max(1, len(SPLICE_THRESHOLDS)//FAST_SWEEP_LEN)]

print('Loading Splice (auto-downloads from UCI if not cached)...')
X_spl, y_spl = load_splice()
print(f'Loaded {X_spl.shape[0]} samples x {X_spl.shape[1]} features, {len(set(y_spl))} classes')

def splice_filter(X, threshold):
    col_counts = np.asarray((X != 0).sum(axis=0)).ravel()
    mask = col_counts >= threshold
    X_f = X[:, mask]
    if X_f.shape[1] == 0:
        return None, {}
    return X_f, {'num_features': int(mask.sum())}

prefix = os.path.join(OUTPUT_DIR, 'splice')
t0 = time.time()

print('Running Splice classification sweep...')
spl_clf = run_classification_experiment(
    X_spl, y_spl, sweep_values=SPLICE_THRESHOLDS, filter_fn=splice_filter,
    dataset_name='Splice', output_prefix=prefix,
    clf_alpha=CLF_ALPHA_PER_DATASET['splice'], cv_folds=CV_FOLDS,
)
plot_experiment_results(
    spl_clf, title='Splice: DNA junction',
    output_prefix=prefix, xlabel='Accuracy',
    xticks=[0.70, 0.80, 0.90, 1.00],
    xtick_labels=['70%','80%','90%','100%'],
    xlim=(0.65, 1.01), ylim=(1e1, 1e5),
)
if SHOW_FIGS: show_pdf(prefix + '_size_vs_accuracy.pdf')

print("\nSplice: No Zhao et al. (2025) baseline — novel Marena 2026 result.")

print('Running Splice PCA sweep...')
spl_pca = run_pca_experiment(
    X_spl, y_spl, sweep_values=SPLICE_THRESHOLDS, filter_fn=splice_filter,
    dataset_name='Splice', output_prefix=prefix, n_components=2,
)
plot_pca_results(spl_pca, title='Splice: Dimension reduction',
    output_prefix=prefix, ylim=(1e1, 1e5))
if SHOW_FIGS: show_pdf(prefix + '_size_vs_variance.pdf')
print(f'Splice done in {time.time()-t0:.0f}s')


## 6. Summary & Output Manifest

In [ ]:
import glob
print('=' * 65)
print('REAL DATASET EXPERIMENTS — OUTPUT MANIFEST')
print('=' * 65)

all_pdfs  = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.pdf')))
all_jsons = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.json')))
print(f'\nPDFs ({len(all_pdfs)}):')
for f in all_pdfs:  print(f'  {f}')
print(f'\nJSON results ({len(all_jsons)}):')
for f in all_jsons: print(f'  {f}')

print('\n' + '='*65)
print('ZHAO ET AL. (2025) REPLICATION SUMMARY')
print('='*65)
print(f"{'Dataset':<14} {'Status':<12} {'Acc Δ':>8} {'Qubits Δ':>10} {'Adv. OOM (ours / Zhao)':>24}")
print('-'*65)
for name in ['imdb', 'news20', 'pbmc68k', 'dorothea']:
    jpath = os.path.join(OUTPUT_DIR, f'{name}_zhao_comparison.json')
    if os.path.exists(jpath):
        with open(jpath) as f: c = json.load(f)
        oom = f"{c['our_advantage_oom']:.1f} / {c['zhao_advantage_oom_range'][0]}–{c['zhao_advantage_oom_range'][1]}"
        print(f"{name:<14} {c['match_status']:<12} {c['accuracy_delta']:>+8.3f} {c['qubit_delta']:>+10d} {oom:>24}")
    else:
        print(f"{name:<14} {'NOT RUN':<12}")
print('-'*65)
print('splice         NOVEL        —          —                      — (no Zhao baseline)')
print('\nReproduction criteria: |Δacc| < 2%, |Δqubits| / zhao_qubits < 20%')
print('Reference: Zhao et al. (2025), Figures 2 and 4 (arXiv:2501.06165)')

print('\nPeak classification accuracy per dataset:')
for name in ['imdb', 'news20', 'pbmc68k', 'dorothea', 'splice']:
    jpath = os.path.join(OUTPUT_DIR, f'{name}_size_vs_accuracy.json')
    if os.path.exists(jpath):
        with open(jpath) as f: d = json.load(f)
        if d.get('accuracies_mean'):
            peak  = max(d['accuracies_mean'])
            min_q = min(d['space_quantum'])
            print(f'  {name:<12}: peak acc={peak:.3f}, min quantum size={min_q} qubits')
    else:
        print(f'  {name:<12}: not run yet')
